# Akamai · Speculative decoding × deterministic FCO/FCG (Lane A)

**One read-only measurement notebook for the hosted Akamai workshop.** Run it now to collect a reference observation from your current `Qwen/Qwen3-4B` deployment. Then follow Akamai's existing **Module 5** (FP8 target) and **Module 6** (speculative drafting) as written, running **this same notebook once after Module 5 and again after Module 6**. It automatically labels the configuration it detects and compares only matched **FP8 non-speculative vs FP8 speculative** runs.

- Source of truth for deployment mutation: [Akamai Module 6](https://github.com/akamai-developers/akamai-workshop-ai-inference/blob/main/06_speculative_decoding/06_speculative_decoding.ipynb). **This notebook never patches Kubernetes.**
- The workshop uses `RedHatAI/Qwen3-4B-FP8-dynamic` with optional `RedHatAI/Qwen3-0.6B-FP8-dynamic` drafter, `num_speculative_tokens=4`.
- Uses workshop's `common.config` and `common.metrics` when available. CPU-only Jupyter pod calls the remote GPU vLLM Service.
- Treat stream chunks as **chunks, not necessarily tokens**. Throughput is from final OpenAI usage `completion_tokens`; no token-level timing is claimed.
- Observations are evidence-linked FCO/FCG projections; SHA-256 proves internal consistency, **not** GPU attestation, signatures or scientific correctness.

### Current evidence ceiling
The previously uploaded `EXP-AKAMAI-TOKEN-001` is a Qwen3-4B (non-FP8) **reference**, not a matched speculative-decoding comparator. Anticube and ΔG* are out of scope until their formula/oracle definitions are frozen.

## 1. Setup — detect workshop modules, model, namespace and serving arguments

In [ ]:
import os, sys, json, hashlib, time, uuid, subprocess, statistics, zipfile
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlsplit
from IPython.display import display, FileLink, Markdown
import requests
import matplotlib.pyplot as plt

MODEL_FP8 = "RedHatAI/Qwen3-4B-FP8-dynamic"
MODEL_ORIGINAL = "Qwen/Qwen3-4B"
DRAFTER = "RedHatAI/Qwen3-0.6B-FP8-dynamic"
REPEATS = 4                     # Bounded read-only workload; increase after first PASS
MAX_TOKENS = 128
TEMPERATURE = 0.0
OUTPUT_ROOT = Path.cwd() / "cell_akamai_results"
OUTPUT_ROOT.mkdir(exist_ok=True)

def sha(data):
    if isinstance(data, str): data = data.encode("utf-8")
    return "sha256:" + hashlib.sha256(data).hexdigest()

def canonical(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False).encode("utf-8")

def cid(kind, payload):
    return sha(canonical({"kind": kind, "payload": payload}))

def command(args, timeout=8):
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
        return p.stdout.strip() if p.returncode == 0 else None
    except (OSError, subprocess.TimeoutExpired):
        return None

def find_workshop(start):
    choices = [start, *start.parents]
    # Also handle a notebook uploaded into a sibling of workshop repo.
    choices += [p for parent in list(start.parents)[:3] for p in parent.glob("*") if p.is_dir()][:100]
    for p in choices:
        if (p/"common"/"config.py").is_file() and (p/"common"/"metrics.py").is_file():
            return p.resolve()
    return None

WORKSHOP_ROOT = find_workshop(Path.cwd())
if WORKSHOP_ROOT:
    sys.path.insert(0, str(WORKSHOP_ROOT))
    from common import config as workshop_config
    from common import metrics as workshop_metrics
    settings = workshop_config.get_settings()
    MODEL_ENV = settings.model_name
    VLLM_HOST = settings.vllm_host
    API_KEY = settings.api_key
    NAMESPACE = settings.namespace
    METRICS_URL = settings.metrics_url
    print("Reusing official workshop config and metrics from:", WORKSHOP_ROOT)
else:
    workshop_metrics = None
    MODEL_ENV = os.getenv("MODEL_NAME", MODEL_ORIGINAL)
    VLLM_HOST = os.getenv("VLLM_HOST", "http://vllm:8000/v1")
    API_KEY = os.getenv("VLLM_API_KEY", "not-needed")
    NAMESPACE = os.getenv("NAMESPACE") or command(["kubectl","config","view","--minify","-o","jsonpath={..namespace}"]) or "UNKNOWN"
    METRICS_URL = VLLM_HOST.rstrip("/").removesuffix("/v1") + "/metrics"
    print("Workshop helpers not found; using identical environment/endpoint contract.")

from openai import OpenAI
client = OpenAI(base_url=VLLM_HOST, api_key=API_KEY, timeout=120)
models = client.models.list()
LIVE_MODEL_IDS = [model.id for model in models.data]
assert LIVE_MODEL_IDS, "No models available"
LIVE_MODEL = LIVE_MODEL_IDS[0]
print("Namespace:", NAMESPACE)
print("Served model:", LIVE_MODEL, "(env hint:", MODEL_ENV + ")")
print("Metrics endpoint:", METRICS_URL)

In [ ]:
deployment_json = command(["kubectl","-n",NAMESPACE,"get","deployment","vllm","-o","json"], timeout=12) if NAMESPACE!="UNKNOWN" else None
deployment = json.loads(deployment_json) if deployment_json else {}
containers = deployment.get("spec",{}).get("template",{}).get("spec",{}).get("containers",[])
live_args = containers[0].get("args",[]) if containers else []
spec_args = [x for x in live_args if x.startswith("--speculative-config")]
if LIVE_MODEL == MODEL_FP8:
    PHASE = "fp8_speculative" if spec_args else "fp8_baseline"
elif LIVE_MODEL == MODEL_ORIGINAL:
    PHASE = "qwen_original_reference"
else:
    PHASE = "other_model_reference"
if PHASE=="fp8_speculative":
    assert DRAFTER in "\n".join(spec_args), "Spec mode detected, but workshop 0.6B drafter identity not verified"
print("PHASE =",PHASE)
print("Spec args =",spec_args or ["NONE"])
print("Deployment observation =", "VERIFIED_BY_KUBECTL" if deployment else "UNKNOWN")
# Do not save deployment environment variables, service accounts or secrets.
# Compare non-spec serving flags without exposing potentially sensitive raw deployment args.
non_spec_args = [a for a in live_args if not a.startswith("--speculative-config")]
runtime_identity = (sha(canonical({"model":LIVE_MODEL, "non_spec_args":non_spec_args,
                                  "container_image":containers[0].get("image","UNKNOWN")}))
                    if deployment else None)
CONFIG_ID = sha(canonical({"served_model": LIVE_MODEL, "spec_args": spec_args,
                           "runtime_identity":runtime_identity, "namespace": NAMESPACE}))

## 2. Exact frozen workload and deterministic arithmetic self-test

In [ ]:
PROMPTS = [
    "Explain KV-cache reuse during autoregressive inference in 4 clear sentences.",
    "Explain prefill and decoding costs in a transformer serving system in 4 clear sentences.",
    "Compare batching and speculative decoding, describing one benefit and one limitation of each.",
    "Explain why GPU memory capacity can limit concurrency in language-model serving.",
]
assert REPEATS <= 8 and REPEATS >= 1
WORKLOAD_ID = sha(canonical({
    "prompts": PROMPTS[:REPEATS], "max_tokens":MAX_TOKENS,
    "temperature":TEMPERATURE,
    "enable_thinking_for_qwen3":False,
    "stream_usage":True,
}))
assert sha(canonical({"b":2,"a":1}))==sha(canonical({"a":1,"b":2}))
assert (20*1e9/2_000_000_000)==10.0
print("Arithmetic / canonicalization self-test: PASS (synthetic fixtures only)")
print("Workload ID:", WORKLOAD_ID)

## 3. Capture actual inference and raw workshop Prometheus measurements

In [ ]:
def metrics_text():
    if workshop_metrics:
        return workshop_metrics.fetch_metrics_text(METRICS_URL,timeout=10)
    response=requests.get(METRICS_URL,timeout=10)
    response.raise_for_status()
    return response.text

def parsed_metrics(text):
    if workshop_metrics:
        return workshop_metrics.parse_metrics(text)
    parsed={}
    for line in text.splitlines():
        if not line or line.startswith("#"): continue
        try:
            left,val=line.split(" ",1)
            key=left.split("{")[0]
            parsed[key]=parsed.get(key,0.0)+float(val.strip())
        except (ValueError, IndexError): pass
    return parsed

def sample_metrics():
    raw=metrics_text()
    return {"raw":raw, "sha256":sha(raw), "parsed":parsed_metrics(raw),
            "captured_utc":datetime.now(timezone.utc).isoformat()}

def metric_value(parsed, name):
    return parsed.get(name)

def kv_fraction(parsed):
    return parsed.get("vllm:kv_cache_usage_perc", parsed.get("vllm:gpu_cache_usage_perc"))

def inter_request_acceleration(records):
    result=[dict(row) for row in records]
    for row in result:
        row["tokens_per_s2"] = None
    for before,after in zip(result,result[1:]):
        dt = (after["midpoint_ns"]-before["midpoint_ns"])/1e9
        if dt>0 and before["tokens_per_s"] is not None and after["tokens_per_s"] is not None:
            after["tokens_per_s2"]=(after["tokens_per_s"]-before["tokens_per_s"])/dt
    return result

RUN_ID = uuid.uuid4().hex[:12]
CREATED_UTC = datetime.now(timezone.utc).isoformat()
observations=[]
prometheus_samples={}
for index in range(REPEATS):
    prompt=PROMPTS[index]
    before=sample_metrics()
    start_ns=time.monotonic_ns()
    first_content_ns=None
    chunk_count=0
    completion_tokens=None
    output_hash=hashlib.sha256()
    stream = client.chat.completions.create(
        model=LIVE_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        stream=True,
        stream_options={"include_usage":True},
        **({"extra_body":{"chat_template_kwargs":{"enable_thinking":False}}} if "qwen3" in LIVE_MODEL.lower() else {})
    )
    for evt in stream:
        stamp_ns=time.monotonic_ns()
        if evt.usage is not None:
            completion_tokens=evt.usage.completion_tokens
        for choice in evt.choices or []:
            delta=choice.delta
            content=(getattr(delta,"reasoning_content",None) or "")+(getattr(delta,"content",None) or "")
            if content:
                chunk_count+=1
                if first_content_ns is None: first_content_ns=stamp_ns
                output_hash.update(content.encode())
    finish_ns=time.monotonic_ns()
    after=sample_metrics()
    duration_ns=finish_ns-start_ns
    rate=completion_tokens*1e9/duration_ns if isinstance(completion_tokens,int) and duration_ns>0 else None
    rec={
        "index":index+1,"prompt_sha256":sha(prompt),
        "response_sha256": "sha256:"+output_hash.hexdigest(),
        "completion_tokens":completion_tokens,"chunk_count":chunk_count,
        "duration_ns":duration_ns, "ttft_first_content_ms":(first_content_ns-start_ns)/1e6 if first_content_ns else None,
        "start_ns":start_ns,"midpoint_ns":(start_ns+finish_ns)/2,
        "tokens_per_s":rate,
        "metrics_before_sha256":before["sha256"],"metrics_after_sha256":after["sha256"],
        "metrics_generated_token_delta": (
            after["parsed"]["vllm:generation_tokens_total"]-before["parsed"]["vllm:generation_tokens_total"]
            if "vllm:generation_tokens_total" in before["parsed"] and "vllm:generation_tokens_total" in after["parsed"] else None
        ),
        "kv_fraction_before":kv_fraction(before["parsed"]),
        "kv_fraction_after":kv_fraction(after["parsed"]),
    }
    observations.append(rec)
    prometheus_samples[f"run{index+1}_before"] = before
    prometheus_samples[f"run{index+1}_after"] = after
    print(f"Run {index+1}: usage tokens={completion_tokens}, {rate:.2f} tokens/s" if rate is not None else f"Run {index+1}: usage UNKNOWN")
observations=inter_request_acceleration(observations)
assert all(r["completion_tokens"] is not None for r in observations), "Missing streaming usage; cannot measure tokens/s reliably"

## 4. Local FCO/FCG evidence projection, reproducible receipt and download links

In [ ]:
status={"mmr":"NOT_COMMITTED","signed":False,"independent_gpu_attestation":"NOT_TESTED",
        "anticube":"NOT_COMPUTED","delta_g_star":"NOT_COMPUTED"}
receipt={
    "schema":"AKAMAI_SPEC_COMPARISON_V1", "experiment_id":"EXP-AKAMAI-SPEC-001",
    "run_id":RUN_ID, "phase":PHASE, "timestamp_utc":CREATED_UTC,
    "workshop_url":"https://github.com/akamai-developers/akamai-workshop-ai-inference",
    "namespace":NAMESPACE,"model":LIVE_MODEL, "model_revision":"UNKNOWN", "tokenizer_revision":"UNKNOWN",
    "deployment_mode":"OBSERVED" if deployment else "UNKNOWN",
    "non_spec_runtime_identity":runtime_identity,
    "speculative_config":spec_args, "config_sha256":CONFIG_ID,
    "workload_id":WORKLOAD_ID,"temperature":TEMPERATURE,"max_tokens":MAX_TOKENS,
    "measurement_scope":"per-request streaming-usage tokens, wall-clock request throughput, inter-request changes in throughput",
    "metrics_scope":"Prometheus snapshots before and after requests, NOT in-flight KV-cache peak",
    "recorded_unit_contract":{"tokens_per_s":"completion tokens / full request seconds",
                              "tokens_per_s2":"difference in per-request tokens/s / midpoint separation seconds",
                              "ttft_first_content_ms":"time to first nonempty chunk (not server-side TTFT)",
                              "kv_fraction":"cache-used fraction when sampled, NOT a percentage"},
    "observations":observations,
    "spec_metric_names_observed":sorted(set(
        key for sample in prometheus_samples.values() for key in sample["parsed"]
        if "spec" in key.lower() and "decode" in key.lower()
    )),
    "states":status
}
directory=OUTPUT_ROOT / f"EXP-AKAMAI-SPEC-001_{PHASE}_{RUN_ID}"
directory.mkdir(parents=True,exist_ok=False)
env_payload={
    "phase":PHASE,"model":LIVE_MODEL,"namespace":NAMESPACE,
    "workload_id":WORKLOAD_ID,"config_sha256":CONFIG_ID
}
atoms=[{"kind":"ExecutionContext","content_id":cid("ExecutionContext",env_payload),
        "occurrence_id":RUN_ID+":env", "payload":env_payload}]
edges=[]
for row in observations:
    obs_id=cid("RequestObservation",row)
    deriv={k:row[k] for k in ["index","tokens_per_s","tokens_per_s2","midpoint_ns"]}
    der_id=cid("TokenKinetics",deriv)
    atoms.extend([{"kind":"RequestObservation","content_id":obs_id,"occurrence_id":RUN_ID+f":{row['index']}", "payload":row},
                  {"kind":"TokenKinetics","content_id":der_id,"occurrence_id":RUN_ID+f":{row['index']}:derived", "payload":deriv}])
    edges.extend([{"source":obs_id,"target":atoms[0]["content_id"],"type":"OBSERVED_IN"},
                  {"source":der_id,"target":obs_id,"type":"DERIVED_FROM"}])
def json_write(path, obj):
    path.write_text(json.dumps(obj,sort_keys=True,indent=2,allow_nan=False)+"\n",encoding="utf-8")
json_write(directory/"experiment_receipt.json", receipt)
(directory/"fco_atoms.jsonl").write_text("".join(json.dumps(a,sort_keys=True,allow_nan=False)+"\n" for a in atoms),encoding="utf-8")
(directory/"fcg_edges.jsonl").write_text("".join(json.dumps(e,sort_keys=True)+"\n" for e in edges),encoding="utf-8")
raw_hash_ok=True
for label,s in prometheus_samples.items():
    p=directory/(label+".prom")
    p.write_text(s["raw"],encoding="utf-8")
    raw_hash_ok &= sha(p.read_bytes())==s["sha256"]
assert raw_hash_ok
assert all(a["content_id"]==cid(a["kind"],a["payload"]) for a in atoms)
ids={a["content_id"] for a in atoms}
assert all(e["source"] in ids and e["target"] in ids for e in edges)
assert all(r["tokens_per_s"]==r["completion_tokens"]*1e9/r["duration_ns"] for r in observations)
assert observations==inter_request_acceleration(observations)
json_write(directory/"verification.json",{"raw_prometheus_hashes":"PASS","fco_content_ids":"PASS",
        "fcg_endpoint_integrity":"PASS","deterministic_derived_arithmetic":"PASS",**status})
archive=OUTPUT_ROOT/(directory.name+".zip")
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(directory.glob("*")):z.write(f,f.name)
print("Receipts:",directory)
print("Independent same-host arithmetic/hash replay: PASS; independent host execution NOT_TESTED.")
print("Download evidence ZIP (authenticated Jupyter link):")
display(FileLink(str(archive.relative_to(Path.cwd()))))

## 5. Results — measured throughput, acceleration and diagnostic deltas

In [ ]:
rates=[r["tokens_per_s"] for r in observations]
times=[(r["midpoint_ns"]-observations[0]["midpoint_ns"])/1e9 for r in observations]
fig,ax=plt.subplots(figsize=(8,3.7))
ax.plot(times,rates,marker="o")
ax.set(xlabel="Elapsed midpoint time (s)",ylabel="Output tokens/s",
       title=f"{PHASE}: observed request throughput ({LIVE_MODEL})")
ax.grid(alpha=0.3)
plt.show()
fig,ax=plt.subplots(figsize=(8,3.7))
ax.plot(times[1:],[r["tokens_per_s2"] for r in observations[1:]],marker="s")
ax.axhline(0,linewidth=1,linestyle="--")
ax.set(xlabel="Elapsed midpoint time (s)",ylabel="Tokens/s²",
       title="Inter-request throughput acceleration (NOT per-token)")
ax.grid(alpha=0.3)
plt.show()
print("Mean tokens/s:",round(statistics.mean(rates),3))
print("Chunk count is not token count. Idle KV-cache samples do not show decode-time peak.")
for r in observations:
    print(f"request {r['index']}: {r['completion_tokens']} tokens, "
          f"{r['tokens_per_s']:.3f} tok/s, {r['ttft_first_content_ms']:.1f} ms first chunk, "
          f"counter_delta={r['metrics_generated_token_delta']}, "
          f"kv_before={r['kv_fraction_before']}, kv_after={r['kv_fraction_after']}")

## 6. Compare ONLY matched FP8 baseline versus FP8 speculative receipts

**No auto-restarts:** If you are currently on Qwen3-4B BF16, first execute Akamai Module 5. Run this notebook once as `fp8_baseline`. Then execute Akamai Module 6 to enable its 0.6B drafter; run this notebook again. Both receipts remain on disk and this cell will build a comparison with derived ratios. The previous `qwen_original_reference` is kept separate.

In [ ]:
all_receipts=[]
for p in sorted(OUTPUT_ROOT.glob("EXP-AKAMAI-SPEC-001_*/experiment_receipt.json")):
    try:
        rr=json.loads(p.read_text(encoding="utf-8"))
        if rr.get("schema")=="AKAMAI_SPEC_COMPARISON_V1" and rr.get("model")==MODEL_FP8:
            all_receipts.append((p,rr))
    except (ValueError,OSError):continue
pair={}
for path,r in all_receipts:
    if r.get("phase") in ("fp8_baseline","fp8_speculative") and r.get("workload_id")==WORKLOAD_ID:
        pair[r["phase"]]=(path,r)  # latest per phase
if len(pair)==2:
    baseline=pair["fp8_baseline"][1]
    speculative=pair["fp8_speculative"][1]
    assert len(baseline["observations"])==len(speculative["observations"])
    control_verified=(baseline.get("non_spec_runtime_identity") is not None
         and baseline["non_spec_runtime_identity"]==speculative.get("non_spec_runtime_identity"))
    if not control_verified:
        print("WARNING: Non-spec runtime identity did not match (or was unavailable).")
        print("Comparison remains descriptive / potentially confounded; inspect target image and flags.")
    before=statistics.mean(x["tokens_per_s"] for x in baseline["observations"])
    after=statistics.mean(x["tokens_per_s"] for x in speculative["observations"])
    delta=(after/before-1)*100
    compare={"schema":"AKAMAI_SPEC_COMPARISON_RESULT_V1",
       "status":"DESCRIPTIVE_OBSERVED_COMPARISON",
       "baseline_receipt_sha256":sha(pair["fp8_baseline"][0].read_bytes()),
       "speculative_receipt_sha256":sha(pair["fp8_speculative"][0].read_bytes()),
       "workload_id":WORKLOAD_ID,"target_model":MODEL_FP8,
       "baseline_mean_tokens_per_s":before,"speculative_mean_tokens_per_s":after,
       "ratio":after/before,"relative_change_percent":delta,
       "non_spec_runtime_identity_equal":control_verified,
       "unit":"completion tokens / full request seconds",
       "limitations":"small n, sequential request performance; no randomization, inference hardware attestation or independent host replay"}
    compare_path=OUTPUT_ROOT/"EXP-AKAMAI-SPEC-001_matched_comparison.json"
    json_write(compare_path,compare)
    display(Markdown(f"**Matched comparison:** {before:.3f} → {after:.3f} tokens/s, "
                     f"ratio **{after/before:.3f}×** ({delta:+.2f}%). "
                     "Observational performance only; neither quality nor causality is established."))
    display(FileLink(str(compare_path.relative_to(Path.cwd()))))
else:
    missing=sorted({"fp8_baseline","fp8_speculative"}-set(pair))
    print("MATCHED_COMPARISON = NOT_TESTED; missing:",missing)
    print("Run Akamai Module 5, this notebook (FP8 baseline), Akamai Module 6, then this notebook again.")

## 7. Interpretation and next action

This notebook records all available model, workload, metrics and calculation identities. A hash addresses data but does not establish observed GPU truth or causal performance effects. **Speculation acceptance metrics** (if present in Prometheus) are retained in raw `.prom` snapshots; their absence is `NOT_AVAILABLE` and must not be replaced with estimates. The next controlled study should repeat baseline/speculative conditions in counterbalanced order, fixed GPU load, and report median/p95 tail latency, verified completion-token throughput and decoding acceptance where instrumented.

**Sharing:** The Jupyter download link works within an authenticated session; it is not automatically a public permalink. To share with another agent, upload sanitized receipts to the private `biobitworks/cell-akamai` repository through a reviewable PR, preserving hashes, original bytes and source identity.